In [295]:
import os
import pickle
import polars as pl
import pandas as pd
import numpy as np

In [296]:
pl.Config.set_tbl_rows(50)
pl.Config.set_fmt_float("full")

polars.config.Config

In [297]:
pd.set_option('display.max_columns', None)

In [298]:
with open('mimiciii_cohort.pkl', 'rb') as file:
    data = pickle.load(file)

/tmpdata/ipykernel_591996/561369422.py:2: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  data = pickle.load(file)


In [299]:
icustays = pl.read_csv('/scratch/sas10092/physionet.org/files/mimiciv/3.1/icu/icustays.csv')
admissions = pl.read_csv('/scratch/sas10092/physionet.org/files/mimiciv/3.1/hosp/admissions.csv')
patients = pl.read_csv('/scratch/sas10092/physionet.org/files/mimiciv/3.1/hosp/patients.csv')

In [300]:
# Index(['', '', '', '', '',
#        '', '', '', '', '',
#        '', '', '', '', '', '', '',
#        'age', 'readmission', 'mortality', 'los_3day', 'los_7day', 'ICD9_CODE',
#        '', '', 'diagnosis'],

In [301]:
ours = pl.read_parquet('../downstream_idx.parquet')

In [302]:
ours = ours.rename({'subject_id':'SUBJECT_ID',
                    'hadm_id':'HADM_ID',
                    'icustay_id':'ICUSTAY_ID',
                    'icu_admission_time':'INTIME',
                    'icu_discharge_time':'OUTTIME',
                    'icu_los_days':'LOS',
                    'mort_24hr_offset':'24h_obs', # not to dana 
                    'mort_48hr_offset':'48h_obs', # note to dana
                    'in_hosp_mort_time':'DOD_HOSP',
                    'out_mortality_time':'DOD_SSN'}) 

In [303]:
ours = ours.drop(['hosp_admission_time', 
                  'hosp_discharge_time',
                  'w24_min', 
                  'w24_max',
                  'w48_min',
                  'w48_max',
                  'wStay_min',
                  'wStay_max',
                  'w24_start_512',
                  'w24_end_512',
                  'w24_start_1024',
                  'w24_end_1024',
                  'w24_start_1536',
                  'w24_end_1536',
                  'w48_start_512',
                  'w48_end_512',
                  'w48_start_1024',
                  'w48_end_1024',
                  'w48_start_1536',
                  'w48_end_1536',
                  'wStay_start_512',
                  'wStay_end_512',
                  'wStay_start_1024',
                  'wStay_end_1024',
                  'wStay_start_1536',
                  'wStay_end_1536',
                  'n_events_hosp',
                  'n_events_icu',
                  'shard',
                  'hosp_los',
                  'hosp_los_hours',
                  'hosp_los_days',
                  'icu_los',
                  'icu_los_hours',])

In [304]:
ours = ours.with_columns(pl.lit('carevue').alias('DBSOURCE'))

In [305]:
icustays = icustays.rename({'subject_id':'SUBJECT_ID',
                            'hadm_id':'HADM_ID',
                            'stay_id': 'ICUSTAY_ID',
                            'first_careunit':'FIRST_CAREUNIT',
                            'last_careunit':'LAST_CAREUNIT'})

In [306]:
icustays = icustays.select(pl.col(['SUBJECT_ID', 'HADM_ID', 'ICUSTAY_ID','FIRST_CAREUNIT','LAST_CAREUNIT']))

In [307]:
ours = ours.join(icustays, on=['SUBJECT_ID', 'HADM_ID', 'ICUSTAY_ID'], how='left')

In [308]:
ours = ours.with_columns(pl.lit(52).alias('FIRST_WARDID'))
ours = ours.with_columns(pl.lit(52).alias('LAST_WARDID'))

In [309]:



patients = patients.with_columns([
    (pl.col("anchor_year") - pl.col("anchor_age")).alias("birth_year")
])

# Step 2: Generate random months (1 to 12) and days (1 to 30)
np.random.seed(42)  # for reproducibility
n = patients.height

random_months = pl.Series("rand_month", np.random.randint(1, 13, size=n))
random_days = pl.Series("rand_day", np.random.randint(1, 29, size=n))

patients = patients.with_columns([
    random_months,
    random_days
])

# Step 3: Combine into string and convert to date
patients= patients.with_columns([
    pl.concat_str([
        pl.col("birth_year").cast(pl.Utf8),
        pl.col("rand_month").cast(pl.Utf8).str.zfill(2),
        pl.col("rand_day").cast(pl.Utf8).str.zfill(2)
    ], separator="-").str.strptime(pl.Date, format="%Y-%m-%d").alias("DOB")
])

In [310]:
patients = patients.rename({'subject_id':'SUBJECT_ID',
                            'gender':'GENDER',
                            'dod':'DOD'
                             }).select(pl.col(['SUBJECT_ID','GENDER','DOB','DOD']))

In [311]:
patients = patients.with_columns(pl.col('DOD').str.strptime(pl.Date, format="%Y-%m-%d"))

In [312]:
patients = patients.with_columns([
    pl.col(['DOB', 'DOD']).cast(pl.Datetime("ns"))
])

In [313]:
ours = ours.join(patients,on='SUBJECT_ID',how='left')

In [314]:
admissions = admissions.rename({'subject_id':'SUBJECT_ID',
                   'hadm_id':'HADM_ID',
                   'hospital_expire_flag':'EXPIRE_FLAG'}).select(pl.col(['SUBJECT_ID','HADM_ID','EXPIRE_FLAG']))

In [315]:
ours = ours.join(admissions, on=['SUBJECT_ID','HADM_ID'], how='left')

In [316]:
ours = ours.with_columns([((pl.col("INTIME") - pl.col("DOB")).dt.total_days() // 365) 
                            .cast(pl.Int32)
                            .alias("age")])

In [324]:
ours = ours[
    "SUBJECT_ID", "HADM_ID", "ICUSTAY_ID", "DBSOURCE", "FIRST_CAREUNIT",
    "LAST_CAREUNIT", "FIRST_WARDID", "LAST_WARDID", "INTIME", "OUTTIME",
    "LOS", "GENDER", "DOB", "DOD", "DOD_HOSP", "DOD_SSN", "EXPIRE_FLAG", 
    "age", "y_mort", "y_mort_1yr", "y_los_7", "y_los_15", "y_los_30",
    "y_icu_readmit", "y_icu_readmit_7", "y_icu_readmit_15", "y_icu_readmit_30",
    "split", '24h_obs','48h_obs'
]

In [329]:
icd = data['ICD9_CODE'][0]

In [330]:
y_icd = data['diagnosis'][0]

In [332]:
ours = ours.with_columns(pl.lit(icd).alias('ICD9_CODE'))

In [334]:
ours = ours.with_columns(pl.lit(y_icd).alias('diagnosis'))

In [337]:
# ours.write_csv('mimiciii_cohort.csv')
print(ours.schema)

Schema([('SUBJECT_ID', Int64), ('HADM_ID', Int64), ('ICUSTAY_ID', Int64), ('DBSOURCE', String), ('FIRST_CAREUNIT', String), ('LAST_CAREUNIT', String), ('FIRST_WARDID', Int32), ('LAST_WARDID', Int32), ('INTIME', Datetime(time_unit='us', time_zone=None)), ('OUTTIME', Datetime(time_unit='us', time_zone=None)), ('LOS', Float64), ('GENDER', String), ('DOB', Datetime(time_unit='ns', time_zone=None)), ('DOD', Datetime(time_unit='ns', time_zone=None)), ('DOD_HOSP', Datetime(time_unit='us', time_zone=None)), ('DOD_SSN', Datetime(time_unit='us', time_zone=None)), ('EXPIRE_FLAG', Int64), ('age', Int32), ('y_mort', Int8), ('y_mort_1yr', Int8), ('y_los_7', Int8), ('y_los_15', Int8), ('y_los_30', Int8), ('y_icu_readmit', Int8), ('y_icu_readmit_7', Int8), ('y_icu_readmit_15', Int8), ('y_icu_readmit_30', Int8), ('split', String), ('24h_obs', Datetime(time_unit='us', time_zone=None)), ('48h_obs', Datetime(time_unit='us', time_zone=None)), ('ICD9_CODE', List(String)), ('diagnosis', List(String))])


In [340]:
ours.to_pandas().to_pickle('mimiciv_cohort.pkl')

In [341]:
from transformers import ModernBertForMaskedLM